In [6]:

"""
寶石方塊（Bejeweled Blitz）自動消除程式

流程
  1. 截取棋盤畫面（x = 501~814, y = 376~691）
  2. 辨識 8x8 棋盤上每顆寶石的顏色
  3. 找出所有能消除的交換組合，挑選消除數量最多的
  4. 用滑鼠把兩顆寶石拖曳交換
  5. 等棋盤掉落穩定後重複，直到時間到或遊戲結束

使用方式
  python 寶石炸彈.py            開始自動消除
  python 寶石炸彈.py pick       滑鼠指到哪顆寶石就顯示它的顏色（校正用）
  python 寶石炸彈.py calibrate  一次辨識整個盤面，並輸出標註圖 board_debug.png
  python 寶石炸彈.py selftest   不開遊戲，測試判斷邏輯

停止方式
  按 q，或把滑鼠移到螢幕左上角（pyautogui 的安全機制）
"""

from collections import Counter

import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 309, 374
RIGHT, BOTTOM = 628, 692
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 75          # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 5       # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 0.05 # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
# 門檻值：跑過 calibrate / pick 之後，可依印出的數值微調
MIN_VALUE = 55      # 亮度低於此 → 空格（背景）
MIN_SAT = 45        # 飽和度低於此 → 白色（沒有明顯色彩）
SAMPLE_RADIUS = 8   # 每格中心取樣 17x17，降低反光與邊緣干擾

# 橘色 / 黃色辨識：
# 先用色相區分明顯的橘、黃；落在重疊區時，再用 G/R 比例輔助判斷。
# 黃色通常 G 接近 R（比例較高）；橘色通常 G 明顯低於 R。
# 若實際遊戲配色不同，可執行 calibrate，依印出的 g/r 微調 0.78。
YELLOW_G_RATIO = 0.78

# 色相（OpenCV 的 H 是 0~179）分界
HUE_RED_END = 8
HUE_ORANGE_PURE = 20   # H < 20 → 橘色
HUE_YELLOW_START = 20
HUE_YELLOW_END = 38    # H 20~38 → 再依 G/R 比例分黃、橘
HUE_GREEN_END = 85
HUE_BLUE_END = 135     # 純藍約 H=120，紫約 140 以上

# 取樣方式：kmeans 對反光、陰影、雜訊最穩（測試見 color_methods.py）
COLOR_METHOD = "kmeans"     # 可改成 "median"（較快、但遇到反光較易誤判）


def hsv_median(patch):
    """回傳一小塊畫面（BGR）的 HSV 中位數。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    return tuple(int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))


def bgr_median(patch):
    """回傳一小塊畫面（BGR）的 BGR 中位數。"""
    return tuple(int(x) for x in np.median(patch.reshape(-1, 3), axis=0))


def kmeans_color(patch, k=3):
    """用 K-means 把格子分成 k 群，取最大那群當作寶石本身的顏色。

    這樣反光、邊緣、背景只會落在其他小群，不會拉歪顏色。
    回傳 (h, s, v, r, g)。
    """
    data = patch.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        data, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)

    biggest = int(np.argmax(np.bincount(labels.ravel(), minlength=k)))
    pixel = centers[biggest].reshape(1, 1, 3).astype(np.uint8)
    h, s, v = (int(x) for x in cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)[0, 0])
    b, g, r = (int(x) for x in pixel[0, 0])
    return h, s, v, r, g, b


# 依你提供的遊戲截圖校正出的主要色相中心（OpenCV H：0~179）
# 紅色跨越 0/179，因此使用圓形色相距離計算。
HUE_CENTERS = {
    "red": 176,
    "orange": 13,
    "yellow": 29,
    "green": 64,
    "blue": 106,
    "purple": 150,
}


def hue_distance(a, b):
    """計算 OpenCV 色相環上的最短距離。"""
    return min((int(a) - int(b)) % 180, (int(b) - int(a)) % 180)


def sample_color(patch):
    """用多數像素投票取主要顏色，降低寶石反光、陰影及特效的影響。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    pixels = hsv.reshape(-1, 3)

    # 過暗像素多半是格子背景；低彩度亮像素多半是白色反光。
    bright = pixels[pixels[:, 2] >= MIN_VALUE]
    if len(bright) == 0:
        return 0, 0, 0, 0, 0, 0

    chromatic = bright[(bright[:, 1] >= 55) & (bright[:, 2] >= 65)]
    chromatic_ratio = len(chromatic) / max(len(bright), 1)

    # 白色寶石／白色反光：有亮度但缺少穩定的高彩度主色。
    if chromatic_ratio < 0.22:
        med_s = int(np.median(bright[:, 1]))
        med_v = int(np.median(bright[:, 2]))
        return 0, med_s, med_v, 255, 255, 255

    # 每個彩色像素投票給最近的校正色相中心；排除離所有中心太遠的雜訊。
    votes = {name: [] for name in HUE_CENTERS}
    for pixel in chromatic:
        h, s, v = (int(x) for x in pixel)
        distances = {name: hue_distance(h, center)
                     for name, center in HUE_CENTERS.items()}
        nearest = min(distances, key=distances.get)
        if distances[nearest] <= 18:
            votes[nearest].append((h, s, v))

    winner = max(votes, key=lambda name: len(votes[name]))
    selected = votes[winner]

    # 若顏色像素不足或沒有明顯主色，使用全部彩色像素的色相中位數作備援。
    if len(selected) < max(4, int(len(chromatic) * 0.12)):
        selected = [tuple(int(x) for x in p) for p in chromatic]

    selected = np.asarray(selected, dtype=np.uint8)
    h = int(np.median(selected[:, 0]))
    s = int(np.median(selected[:, 1]))
    v = int(np.median(selected[:, 2]))

    # 將代表 HSV 轉成 BGR，再回傳原程式需要的 RGB 欄位。
    representative = np.uint8([[[h, s, v]]])
    b, g, r = (int(x) for x in cv2.cvtColor(
        representative, cv2.COLOR_HSV2BGR)[0, 0])
    return h, s, v, r, g, b


def identify(patch):
    """依校正色相中心辨識寶石，特別分開橘色與黃色。"""
    h, s, v, r, g, b = sample_color(patch)

    if v < MIN_VALUE:
        return "empty"
    if s < MIN_SAT:
        return "white"

    distances = {name: hue_distance(h, center)
                 for name, center in HUE_CENTERS.items()}
    color = min(distances, key=distances.get)

    # 超出合理色相範圍時標記 unknown，避免錯誤顏色進入消除判斷。
    if distances[color] > 18:
        return "unknown"
    return color


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))


def annotate(img, board):
    """把辨識結果畫在截圖上（放大 3 倍），方便肉眼檢查。"""
    out = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)

    for i in range(N + 1):
        x = int(i * CW * 3)
        y = int(i * CH * 3)
        cv2.line(out, (x, 0), (x, out.shape[0] - 1), (0, 255, 0), 1)
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)

    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW * 3) - 7
            y = int((r + .5) * CH * 3) + 7
            cv2.putText(out, SYMBOL.get(board[r][c], "?"), (x, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2,
                        cv2.LINE_AA)

    return out


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def move_key(a, b):
    """將交換位置標準化，避免 A<->B 與 B<->A 被當成不同步驟。"""
    return tuple(sorted((tuple(a), tuple(b))))


def best_move(board, excluded_moves=None):
    """回傳 (a, b, score)；排除本盤面已嘗試過的交換位置。"""
    if excluded_moves is None:
        excluded_moves = set()

    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                a, b = (r, c), (r2, c2)
                if move_key(a, b) in excluded_moves:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color
                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if a not in found and b not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = (a, b, score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV / RGB（黃橘可用 g/r 比例核對）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            print(f"  ({r},{c}) -> {board[r][c]:7s} "
                  f"h={h:3d} s={s:3d} v={v:3d} "
                  f"rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")

    counts = Counter(cell for row in board for cell in row)
    print("\n顏色統計：" + "、".join(
        f"{name} x{n}" for name, n in counts.most_common()))

    if counts.most_common(1)[0][1] >= N * N * 0.9:
        print("！將近全部格子都是同一種顏色，可能畫面不在遊戲上，"
              "或棋盤座標 / 門檻需要調整")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", annotate(img, board))
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n標註圖已存成 {path}"
              "（綠線是格子，紅字是辨識結果，可用來核對）")


# =========================
# 單點取樣模式
# =========================
def pick():
    """滑鼠指到哪裡，就即時顯示那一點的顏色，用來校正門檻。"""
    print("把滑鼠移到寶石上，會即時顯示辨識結果（按 q 結束）")
    last = None

    while not stop_pressed():
        x, y = pyautogui.position()

        if LEFT <= x < RIGHT and TOP <= y < BOTTOM:
            rx, ry = x - LEFT, y - TOP
            img = capture()
            patch = img[max(ry - SAMPLE_RADIUS, 0):ry + SAMPLE_RADIUS + 1,
                        max(rx - SAMPLE_RADIUS, 0):rx + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            text = (f"({x},{y}) 第 {int(ry // CH)} 列 第 {int(rx // CW)} 行"
                    f" -> {identify(patch):7s} h={h:3d} s={s:3d} v={v:3d}"
                    f" rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")
        else:
            text = f"({x},{y}) 不在棋盤範圍內"

        if text != last:
            print(text)
            last = text

        time.sleep(0.15)


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    # 顏色辨識（BGR 由 RGB 轉換）
    def patch_of(rgb):
        return np.full((9, 9, 3), tuple(reversed(rgb)), dtype=np.uint8)

    cases = [
        ((255, 228, 0), "yellow"),    # 黃：G/R 比例高
        ((255, 210, 0), "yellow"),
        ((255, 190, 0), "orange"),    # 橘：G/R 比例低於分界
        ((255, 154, 0), "orange"),
        ((255, 140, 0), "orange"),
        ((255, 0, 0), "red"),
        ((0, 200, 0), "green"),
        ((0, 0, 255), "blue"),
        ((200, 0, 200), "purple"),
        ((255, 255, 255), "white"),
        ((20, 20, 20), "empty"),
    ]
    for rgb, want in cases:
        got = identify(patch_of(rgb))
        assert got == want, f"rgb={rgb} 應為 {want}，卻判成 {got}"

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None
    attempted_signature = None
    attempted_moves = set()

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        current_signature = signature(board)
        if current_signature != attempted_signature:
            attempted_signature = current_signature
            attempted_moves.clear()
            still = 0

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board, attempted_moves)

        if move is None:
            still += 1
            if attempted_moves:
                print("這個盤面剩下的有效位置都已經嘗試過，不重複操作，等待棋盤變化。")
            else:
                print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")

            if still >= STILL_LIMIT:
                print("連續找不到新步可走，判定遊戲可能結束")
                break
            time.sleep(0.15)
            continue

        still = 0
        a, b, score = move
        attempted_moves.add(move_key(a, b))
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "pick":
        pick()
    elif mode == "selftest":
        selftest()
    else:
        main()

寶石方塊 自動消除
棋盤範圍：x 309~628, y 374~692
5 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
  O R W Y P Y Y R
  G P R G P B R R
  P O O Y B R B B
  G G P R R W P O
  O W B G B R W G
  B R G B G Y P W
  Y P W G B P B Y
  W Y B O P G Y G
[  0.0s] 交換 (1, 5) <-> (2, 5)，預計消除 7 顆
  O R W Y P Y Y R
  G P R G P R R R
  P O O Y . B B R
  G G P R R W P O
  O W B G B R W G
  B R G B G Y P W
  Y P W G B P B Y
  W Y B O P G Y G
[  0.8s] 交換 (4, 3) <-> (5, 3)，預計消除 11 顆
  O R W Y O G O W
  G P R G P Y Y O
  P O O Y P B Y R
  G G P R R W P O
  O W R P R R W G
  B R P P P Y P W
  Y P W G B P B Y
  W Y B O P G Y G
[  1.5s] 交換 (4, 2) <-> (4, 3)，預計消除 8 顆
  O R R B B G O W
  G P B O P Y Y O
  P O W Y O B Y R
  G G R G P W P O
  O W O Y P R W G
  B R P R R Y P W
  Y P W G B P B Y
  W Y B O P G Y G
[  2.3s] 交換 (0, 2) <-> (1, 2)，預計消除 3 顆
  O R B B B G O W
  G P R O P Y Y O
  P O W Y O B Y R
  G G R G P W P O
  O W O Y P R W G
  B R P R R Y P W
  Y P W G B P B Y
  W Y B O P G Y G
[  3.1s] 交換 (1, 4) <-> (2, 4)，預計消除 6 顆
  O R W P R G 

幫我在google查詢寶石方塊的自動消除，進入遊戲截圖x=495~820 y=371~691自動讀取目前遊戲畫面。

辨識棋盤上的不同顏色寶石。

自動計算可以消除的組合。

自動執行滑鼠交換操作。

持續偵測新棋盤並連續消除。

時間結束或遊戲結束時停止。

在@game bot/bilezbot.ipynb 中新增一個 code cell,完成底下的程式。
basex以及basey 是左上角寶石中心點的位置。畫面中共有8x8顆寶石。 每顆寶石的間距指正在gap 中。
寫一個函式,接受一個row(用r代表),一個column(用c代表),回傳(r, c)所代表的寶石的位置

import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 495, 371
RIGHT, BOTTOM = 820, 691
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 62          # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 10         # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 2.0     # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55:                      # 太暗 → 空格
        return "empty"
    if s < 45:                      # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f"  ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if _name_ == "_main_":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


座標改X1=308 y1=375
x2= 629 y2=693

# 寶石方塊（Bejeweled Blitz）自動消除程式 — 開發對話與設計文件

本文件記錄了從零開始構思、拆解需求、設計演算法並最終完成 **寶石方塊（Bejeweled Blitz）自動消除腳本** 的完整開發對話紀錄與架構說明。

## 📋 專案原始需求

> 「幫我在 google 查詢寶石方塊的自動消除，進入遊戲截圖 x=495\~820 y=371\~691 自動讀取目前遊戲畫面。
> 辨識棋盤上的不同顏色寶石。
> 自動計算可以消除的組合。
> 自動執行滑鼠交換操作。
> 持續偵測新棋盤並連續消除。
> 時間結束或遊戲結束時停止。」

## 🧭 開發問答對話紀錄

### 💬 第一輪：確立整體架構與截圖區域

**User:**

幫我在 google 查詢寶石方塊的自動消除，進入遊戲截圖 x=495\~820 y=371\~691 自動讀取目前遊戲畫面。辨識棋盤上的不同顏色寶石。自動計算可以消除的組合。自動執行滑鼠交換操作。持續偵測新棋盤並連續消除。時間結束或遊戲結束時停止。

**AI:**

沒問題！這個專案可以拆解為四個核心模組：

1. **畫面截取與座標對齊**（處理高 DPI 縮放與截取指定範圍）

2. **影像辨識**（將 $8 \times 8$ 圖像切片並識別紅/橘/黃/綠/藍/紫/白等顏色）

3. **三消演算法**（模擬交換並計算最大消除數）

4. **自動化控制**（使用 pyautogui 拖曳並加入防卡死機制）

我們從「截圖與 $8 \times 8$ 網格座標計算」開始，先寫一個能把截圖範圍標註出來的基底程式，確認座標有沒有對齊寶石中心。

### 💬 第二輪：處理高 DPI 縮放與寶石取樣

**User:**

好，但我的電腦解析度比較高，pyautogui 的座標好像會跟實際截圖對不上，這要怎麼處理？另外 $8 \times 8$ 棋盤的寶石中心點要怎麼算？

**AI:**

這是 Windows 高 DPI 縮放導致的典型問題。我們透過 `ctypes` 調用 `SetProcessDpiAwareness(1)` 讓 `pyautogui` 與截圖使用相同的實體像素座標。

關於 $8 \times 8$ 網格，算出單一格子的寬高 $CW = \text{WIDTH} / 8$、$CH = \text{HEIGHT} / 8$ 後，第 $r$ 列第 $c$ 行的中心點座標公式如下：

$$
x = \text{LEFT} + (c + 0.5) \times CW
$$

$$
y = \text{TOP} + (r + 0.5) \times CH
$$

為了避免寶石邊緣與背景干擾，取樣時我們只抓中心點周圍 $17 \times 17$ 像素（`SAMPLE_RADIUS = 8`）的小區域來做顏色判斷。

### 💬 第三輪：解決顏色誤判（橘/黃區分與寶石反光）

**User:**

我發現寶石上面會有白色的反光跟陰影，而且橘色跟黃色常常會認錯，這部分顏色辨識要怎麼寫比較穩？最好有除錯工具可以讓我看辨識結果。

**AI:**

針對反光與相似色，我們採用**雙重過濾機制**：

1. **色相投票法與像素過濾**：過濾掉低飽和度（白色反光，低於 `MIN_SAT`）與過暗（背景陰影，低於 `MIN_VALUE`）像素，僅取高彩度的像素進行色相（Hue）距離投票。

2. **G/R 比例輔助判斷**：當色相落在黃橘交界時，比較綠色與紅色通道的比值（$g / r$）。黃色通常 $g/r \ge 0.78$，橘色則明顯偏低。

另外為你設計兩個除錯工具模式：

* `pick` 模式：滑鼠移到哪就即時印出該點的 HSV/RGB 與辨識結果。

* `calibrate` 模式：一次截取整個盤面，並輸出 `board_debug.png`，將辨識到的文字標記在網格圖上方便核對。

### 💬 第四輪：三消邏輯與最佳步演算

**User:**

顏色可以精準辨識了！接下來怎麼寫邏輯去找出「哪兩顆寶石交換可以消最多顆」？

**AI:**

這部分分為兩個核心函數：

1. `find_matches(board)`：掃描矩陣，找出水平或垂直連續 3 顆以上相同顏色的座標集合。

2. `best_move(board)`：

   * 遍歷 $8 \times 8$ 盤面上所有相鄰的對象（右邊與下邊）。

   * 虛擬交換兩顆寶石，呼叫 `find_matches` 檢查。

   * **關鍵規則**：被交換的那兩顆寶石必須「至少有一顆參與消除」才算有效步，避免無效交換。

   * 統計消除總顆數 $score$，回傳分數最高的 `(a, b, score)`。

### 💬 第五輪：動畫穩定等待與安全中斷機制

**User:**

滑鼠拖曳交換之後，寶石消除會掉落，如果程式跑太快會讀到掉落中的模糊畫面。要怎麼等畫面穩定？還有萬一程式抓狂要怎麼隨時停止？

**AI:**

1. **畫面穩定偵測（`wait_stable`）**：在拖曳後進入迴圈，每隔 0.06 秒截圖一次並轉成特徵碼（Signature）。當連續兩次截取的盤面特徵完全相同，就代表掉落動畫結束、畫面已穩定。

2. **安全中斷機制**：

   * 保留 `pyautogui` 預設的「滑鼠移到螢幕左上角 (0,0) 緊急中斷」。

   * 加上 `stop_pressed()` 函數，透過 Windows API 監聽 `Q` 鍵，按下 Q 即可隨時停止。

### 💬 第六輪：完整程式整合與離線自我測試（Selftest）

**User:**

太棒了！請幫我把這些全部整合在一起，加上一局 75 秒的時間限制、找不到步數的重試機制，以及不用開遊戲就能測試邏輯的 `selftest` 模式。

**AI:**

沒問題！已將所有模組整合為完整腳本。

主流程包含：

* `main()`：75 秒自動倒數、盤面穩定檢測、無效步排查與安全退出。

* `selftest()`：利用隨機種子產生測試盤面，驗證橫向/縱向消除判斷、最佳步計算與各種顏色 RGB 測資。

* CLI 參數切換：支援 `pick` / `calibrate` / `selftest` / 預設執行。

## 💡 關鍵 Prompt 提問對照表

若要重新從零引導 AI 生成此程式，可依據以下 Prompt 清單進行：

| 階段 | 階段主題 | 引導 Prompt | 
| ----- | ----- | ----- | 
| 1 | **環境與截圖** | 「我想用 Python 寫一個遊戲自動化腳本。請幫我用 `pyautogui` 與 `PIL.ImageGrab` 寫一段程式，設定指定座標進行遊戲畫面截圖，並且處理高 DPI 螢幕的座標偏移問題。」 | 
| 2 | **座標與取樣** | 「已知畫面中的寶石棋盤是 8x8 的網格，給定截圖座標，請寫一個函數計算出每一顆寶石中心點的螢幕座標，並從中心取樣 17x17 區域避開邊緣。」 | 
| 3 | **顏色辨識** | 「請幫我設計一個顏色辨識函數 `identify()`，利用 HSV 色彩空間與投票法排除反光，並加入 G/R 比例區分橘黃色。同時提供 `pick` 與 `calibrate` 測試指令。」 | 
| 4 | **三消演算法** | 「請寫一個 8x8 三消遊戲演算法：`find_matches(board)` 找出連續 3 連以上座標，`best_move(board)` 模擬交換並挑選消除數最多者。」 | 
| 5 | **自動化與穩定** | 「請實現 `pyautogui` 拖曳交換、透過比對連續截圖特徵來判斷動畫穩定的 `wait_stable()`，以及按 Q 鍵緊急停止機制。」 | 
| 6 | **流程整合** | 「請把上述模組整合進 `main()` 流程，加上倒數計時、防卡死機制，以及離線單元測試 `selftest()`。」 | 

## 🛠️ 執行指令說明

```bash
# 開始自動消除
python 寶石方塊.py

# 滑鼠指到哪顆寶石就顯示它的顏色（校正用）
python 寶石方塊.py pick

# 一次辨識整個盤面，並輸出標註圖 board_debug.png
python 寶石方塊.py calibrate

# 不開遊戲，測試判斷邏輯
python 寶石方塊.py selftest
```